# Eigendecomposition — companion notebook

Runnable companion for [`eigendecomposition.md`](./eigendecomposition.md).

1. Compute eigendecomposition of a small symmetric matrix.
2. Visualize eigenvectors as the directions a 2x2 transform leaves invariant.
3. Power iteration converging to the dominant eigenvector.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Symmetric 2x2 matrix

$$ A = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} $$

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])

# eigh: symmetric solver, returns real eigenvalues sorted ascending
lam, Q = np.linalg.eigh(A)
print('eigenvalues =', lam)
print('eigenvectors (columns of Q) =\n', Q)

# Reconstruct: A = Q diag(lam) Q^T
print('\nQ @ diag(lam) @ Q.T =\n', Q @ np.diag(lam) @ Q.T)

# Sanity checks against trace/determinant
print(f'\ntrace(A) = {np.trace(A):.4f}   sum(lam) = {lam.sum():.4f}')
print(f'det(A)   = {np.linalg.det(A):.4f}   prod(lam) = {np.prod(lam):.4f}')

## 2. Invariant directions

Plot the unit circle, its image under $A$ (an ellipse), and the eigenvectors. Eigenvectors map onto themselves (up to scale $\lambda_i$), unlike a generic input which gets rotated.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(theta), np.sin(theta)], axis=0)
image = A @ circle

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(circle[0], circle[1], 'tab:blue', alpha=0.6, label='unit circle')
ax.plot(image[0], image[1], 'tab:orange', alpha=0.6, label='A @ circle')

colors = ['tab:red', 'tab:green']
for i in range(2):
    v = Q[:, i]
    Av = A @ v
    ax.annotate('', xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=colors[i], lw=2))
    ax.annotate('', xy=Av, xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=colors[i], lw=2, ls='dashed'))
    ax.text(*(Av * 1.1), f'A v{i+1} = {lam[i]:.2f} v{i+1}', color=colors[i])

ax.set_aspect('equal'); ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.legend(loc='lower right')
ax.set_title('Eigenvectors are invariant directions (solid = v, dashed = Av)')
plt.show()

## 3. Power iteration

Repeatedly applying $A$ and normalizing drives a random starting vector toward the dominant eigenvector (the one with largest $|\lambda|$), with the Rayleigh quotient converging to $\lambda_{\max}$.

In [ ]:
# Use a larger symmetric matrix to make the demo more interesting
n = 5
M = rng.standard_normal((n, n))
B = M @ M.T + 0.1 * np.eye(n)  # symmetric PSD

lam_true, V_true = np.linalg.eigh(B)
v_dom = V_true[:, -1]            # eigenvector for largest eigenvalue
lam_dom = lam_true[-1]

x = rng.standard_normal(n)
x /= np.linalg.norm(x)

n_iter = 30
errs, rayleigh = [], []
for _ in range(n_iter):
    x = B @ x
    x /= np.linalg.norm(x)
    rq = x @ B @ x
    # Sign-invariant error: angle-like distance
    errs.append(min(np.linalg.norm(x - v_dom), np.linalg.norm(x + v_dom)))
    rayleigh.append(rq)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(errs, marker='o', ms=4)
axes[0].set_xlabel('iteration'); axes[0].set_ylabel('||x - v_dom||  (log)')
axes[0].set_title('Convergence to dominant eigenvector')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].plot(rayleigh, marker='o', ms=4, label='Rayleigh quotient')
axes[1].axhline(lam_dom, color='tab:red', ls='--', label=f'true lambda_max = {lam_dom:.4f}')
axes[1].set_xlabel('iteration'); axes[1].set_ylabel('x^T B x')
axes[1].set_title('Rayleigh quotient -> lambda_max')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'final Rayleigh quotient = {rayleigh[-1]:.6f}')
print(f'true lambda_max        = {lam_dom:.6f}')
# Convergence ratio is |lambda_{n-1}/lambda_n| per step
print(f'predicted ratio per step = {lam_true[-2]/lam_true[-1]:.4f}')

## 4. Defective matrix pitfall

A Jordan block has fewer eigenvectors than its dimension and cannot be diagonalized.

In [ ]:
J = np.array([[1.0, 1.0],
              [0.0, 1.0]])
lam, V = np.linalg.eig(J)
print('eigenvalues:', lam)
print('eigenvectors (columns):\n', V)
print('rank of eigenvector matrix =', np.linalg.matrix_rank(V),
      ' (would need 2 for diagonalization)')